# Correct and Smooth (C&S) Post-Processing on Cora

Node Classification on Cora (Planetoid): Combining simple base MLP predictions with graph error-correction and smoothing. This notebook implements the approach with `CorrectAndSmooth` inside a `BaseMLP` model, trained with the Adam optimizer for 50 epochs, evaluating the result on held-out data. The single code cell below installs **K3-Node**, loads the dataset, defines the model using K3-Node's `CorrectAndSmooth` on **Keras 3**, compiles and trains it, and reports the resulting metric — the same code runs unchanged on the PyTorch, TensorFlow, or JAX backend by switching the `KERAS_BACKEND` environment variable.

In [ ]:
# Setup environment and install dependencies
!pip install -q torch_geometric
!pip install git+http://github.com/anas-rz/k3-node/@examples-check

# ==============================================================================
# K3-Node (Keras 3 Multi-Backend) Implementation
# ==============================================================================
import os
os.environ.setdefault("KERAS_BACKEND", "tensorflow")

import keras
from keras import layers, ops

import k3_node
from k3_node import layers as k3_layers
from k3_node import models as k3_models
from k3_node.datasets import Planetoid
from k3_node import transforms as k3_transforms

title = "Correct and Smooth (C&S) Post-Processing Pipeline"
backend = keras.config.backend()
print(f"[K3-Node] Initializing {title} on Keras 3 ({backend}) backend...")

# 1. Dataset
dataset = Planetoid(root="./data/Planetoid", name="Cora", transform=k3_transforms.NormalizeFeatures())
data = dataset[0]
num_features = dataset.num_features
num_classes = dataset.num_classes

# 2. Base MLP Model
class BaseMLP(keras.Model):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.lin1 = layers.Dense(hidden_channels, activation="relu")
        self.lin2 = layers.Dense(out_channels)
        self.dropout = layers.Dropout(0.5)

    def call(self, x, training=False):
        x = self.dropout(x, training=training)
        x = self.lin1(x)
        x = self.dropout(x, training=training)
        return self.lin2(x)

base_model = BaseMLP(num_features, 64, num_classes)
base_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.01),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    weighted_metrics=[keras.metrics.SparseCategoricalAccuracy(name="acc")],
)

# 3. Train Base Model
def data_gen():
    mask = ops.cast(data.train_mask, "float32")
    while True:
        yield data.x, data.y, mask

print("Training Base MLP...")
base_model.fit(data_gen(), steps_per_epoch=1, epochs=50, verbose=0)

# 4. Evaluate Base Predictions
logits = base_model(data.x)
base_pred = ops.argmax(logits, axis=-1)
base_acc = float(ops.mean(ops.cast(ops.cast(base_pred[data.test_mask], "int64") == ops.cast(data.y[data.test_mask], "int64"), "float32")))
print(f"Base MLP Test Accuracy: {base_acc:.4f}")

# 5. Correct and Smooth Post-Processing
post = k3_models.CorrectAndSmooth(
    num_correction_layers=50,
    correction_alpha=0.8,
    num_smoothing_layers=50,
    smoothing_alpha=0.8,
    autoscale=False,
    scale=1.0,
)

y_one_hot = ops.one_hot(ops.cast(data.y, "int32"), num_classes)
y_soft = ops.softmax(logits, axis=-1)

cs_out = post.correct(y_soft, y_one_hot[data.train_mask], data.train_mask, data.edge_index)
cs_out = post.smooth(cs_out, y_one_hot[data.train_mask], data.train_mask, data.edge_index)

cs_pred = ops.argmax(cs_out, axis=-1)
cs_acc = float(ops.mean(ops.cast(ops.cast(cs_pred[data.test_mask], "int64") == ops.cast(data.y[data.test_mask], "int64"), "float32")))
print(f"Post-Processed (C&S) Test Accuracy: {cs_acc:.4f}")

print("\n✓ K3-Node C&S execution completed successfully!")